# 05 Experiments

This notebook runs both experiment paths in the project:

- the production model-selection workflow used by training and deployment
- the research-aligned benchmark that studies turbulence, volatility, and hybrid behavior


## Research Paper Alignment

The production workflow optimizes a deployable model on engineered features. The research workflow is broader: it compares linear, nonlinear, and hybrid forecasting behavior under regime-sensitive diagnostics inspired by the paper.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 130

ASSET_DIR = PROJECT_ROOT / "presentation" / "figures"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [ ]:
{
    "production_experiments": params["experiments"],
    "research_experiments": params["research_experiments"],
}


In [ ]:
subprocess.run([sys.executable, "dvc_pipeline/src/run_experiments.py"], check=True)
subprocess.run([sys.executable, "dvc_pipeline/src/update_params_from_experiments.py"], check=True)


In [ ]:
results_df = pd.read_csv(params["experiments"]["results_path"]).sort_values("test_rmse").reset_index(drop=True)
best_result = read_json(params["experiments"]["best_result_path"])
updated_params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))

results_df, best_result, updated_params["training"]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

sns.barplot(
    data=results_df,
    x="test_rmse",
    y="model_name",
    hue="model_name",
    dodge=False,
    legend=False,
    ax=axes[0],
    palette="crest",
)
axes[0].set_title("Production Model Comparison")
axes[0].set_xlabel("Test RMSE")
axes[0].set_ylabel("")

long_df = results_df.melt(
    id_vars="model_name",
    value_vars=["train_rmse", "test_rmse"],
    var_name="split",
    value_name="rmse",
)
sns.barplot(data=long_df, x="model_name", y="rmse", hue="split", ax=axes[1], palette="Set2")
axes[1].set_title("Generalization Check")
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_xlabel("")

fig.tight_layout()
fig.savefig(ASSET_DIR / "05_production_model_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
subprocess.run([sys.executable, "dvc_pipeline/src/research_experiments.py"], check=True)


In [ ]:
research_cfg = params["research_experiments"]
research_results = pd.read_csv(research_cfg["results_path"]).sort_values("rmse").reset_index(drop=True)
research_summary = read_json(research_cfg["summary_path"])
research_predictions = pd.read_csv(research_cfg["predictions_path"])
arima_lstm_history = read_json(research_cfg["arima_lstm"]["history_path"])
history_df = pd.DataFrame(arima_lstm_history["history"])

research_results, research_summary, history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

sns.barplot(
    data=research_results,
    x="rmse",
    y="model_name",
    hue="model_name",
    dodge=False,
    legend=False,
    ax=axes[0],
    palette="mako",
)
axes[0].set_title("Research Benchmark: RMSE")
axes[0].set_xlabel("RMSE")
axes[0].set_ylabel("")

regime_df = research_results.melt(
    id_vars="model_name",
    value_vars=["rmse_high_volatility", "rmse_low_volatility"],
    var_name="regime",
    value_name="regime_rmse",
)
sns.barplot(data=regime_df, x="model_name", y="regime_rmse", hue="regime", ax=axes[1], palette="flare")
axes[1].set_title("Volatility-Regime Error")
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_xlabel("")

fig.tight_layout()
fig.savefig(ASSET_DIR / "06_research_benchmark.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))

sns.lineplot(data=history_df, x="epoch", y="train_loss", ax=ax, label="Train loss", linewidth=1.8)
sns.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax, label="Validation loss", linewidth=1.8)
ax.axvline(arima_lstm_history["best_epoch"], color="#7c3aed", linestyle="--", linewidth=1.0, label="Best epoch")
ax.set_title("ARIMA + LSTM Residual Training Curve")
ax.set_ylabel("MSE loss")
ax.legend()

fig.tight_layout()
fig.savefig(ASSET_DIR / "14_arima_lstm_history.png", bbox_inches="tight")
plt.show()

In [ ]:
trace_df = research_predictions.copy()
trace_df["Date"] = pd.to_datetime(trace_df["Date"])
trace_df = trace_df.tail(80)

fig, ax = plt.subplots(figsize=(13, 5.4))
sns.lineplot(data=trace_df, x="Date", y="actual", ax=ax, label="Actual", linewidth=2.2, color="black")
sns.lineplot(data=trace_df, x="Date", y="pred_arima", ax=ax, label="ARIMA", linewidth=1.4)
sns.lineplot(data=trace_df, x="Date", y="pred_svr_rbf", ax=ax, label="SVR-RBF", linewidth=1.5)
sns.lineplot(
    data=trace_df,
    x="Date",
    y="pred_hybrid_arima_lstm",
    ax=ax,
    label="Hybrid ARIMA + LSTM",
    linewidth=1.8,
)
ax.set_title("Prediction Trace on the Test Window")
ax.set_ylabel("Price")

fig.tight_layout()
fig.savefig(ASSET_DIR / "07_research_prediction_traces.png", bbox_inches="tight")
plt.show()